In [ ]:
"""
CryptoTick HitBTC Pipeline
===========================
Downloads and incrementally updates tick-level trade data from HitBTC's
public REST API v3, storing it as one Parquet file PER COIN PER DAY.

Why partition by day instead of one giant file per coin?
  - Parquet files cannot be appended to in place. Partitioning by day means
    each day is written exactly once and never touched again, which makes
    resuming after a crash trivial (just skip days whose file already exists)
    and makes daily updates trivial (just add today's file).

Usage:
    python hitbtc_pipeline.py backfill   # one-time historical backfill
    python hitbtc_pipeline.py update     # daily incremental update (run via cron / GitHub Actions)
"""

import requests
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import time
import os
import sys
import logging
from datetime import datetime, timedelta, timezone

# ----------------------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------------------

BASE_URL = "https://api.hitbtc.com/api/3/public/trades/{symbol}"
SYMBOL_LIST_URL = "https://api.hitbtc.com/api/3/public/symbol"

COINS = ["ETH", "BTC", "BCH", "BSV", "LTC", "XMR", "ZEC", "DASH"]
QUOTE_CANDIDATES = ["USDT", "USD"]  # tried in order per coin

START_DATE = datetime(2019, 11, 1, tzinfo=timezone.utc)

OUTPUT_DIR = "data/HitBTC_Intraday_Data"
LOG_DIR = "logs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

MAX_LIMIT = 1000
REQUEST_SLEEP = 0.25
MAX_RETRIES = 5

SCHEMA = pa.schema([
    ("id", pa.int64()),
    ("price", pa.float64()),
    ("qty", pa.float64()),
    ("side", pa.string()),
    ("timestamp", pa.timestamp("ms", tz="UTC")),
    ("coin", pa.string()),
])

# ----------------------------------------------------------------------------
# Logging
# ----------------------------------------------------------------------------

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(os.path.join(LOG_DIR, "hitbtc_pipeline.log")),
        logging.StreamHandler(sys.stdout),
    ],
)
log = logging.getLogger("hitbtc_pipeline")

# ----------------------------------------------------------------------------
# Symbol validation
# ----------------------------------------------------------------------------

def resolve_symbol(coin, session):
    """
    Confirms which quote currency actually has a live trading pair for this
    coin on HitBTC. Tries USDT first, then USD. Returns None if neither exists.
    """
    try:
        resp = session.get(SYMBOL_LIST_URL, timeout=15)
        resp.raise_for_status()
        available = set(resp.json().keys())
    except Exception as e:
        log.warning(f"Could not fetch symbol list ({e}); assuming USDT pair exists.")
        return f"{coin}USDT"

    for quote in QUOTE_CANDIDATES:
        candidate = f"{coin}{quote}"
        if candidate in available:
            return candidate

    log.error(f"No USDT/USD pair found for {coin} on HitBTC. Skipping.")
    return None

# ----------------------------------------------------------------------------
# Core fetch logic
# ----------------------------------------------------------------------------

def fetch_trades_for_window(symbol, from_ts_ms, till_ts_ms, session):
    """
    Fetch all trades for a symbol between from_ts_ms and till_ts_ms (inclusive),
    paginating forward using the timestamp of the last trade received.
    Retries transient failures with exponential backoff.
    """
    all_trades = []
    current_from = from_ts_ms
    retries = 0

    while True:
        params = {
            "sort": "ASC",
            "by": "timestamp",
            "from": current_from,
            "till": till_ts_ms,
            "limit": MAX_LIMIT,
        }
        try:
            resp = session.get(BASE_URL.format(symbol=symbol), params=params, timeout=30)
        except requests.RequestException as e:
            retries += 1
            if retries > MAX_RETRIES:
                log.error(f"{symbol}: giving up after {MAX_RETRIES} retries ({e})")
                break
            wait = 2 ** retries
            log.warning(f"{symbol}: network error ({e}), retrying in {wait}s...")
            time.sleep(wait)
            continue

        if resp.status_code == 429:
            log.warning(f"{symbol}: rate limited, sleeping 5s...")
            time.sleep(5)
            continue

        if resp.status_code != 200:
            log.error(f"{symbol}: HTTP {resp.status_code} at {current_from}: {resp.text[:200]}")
            break

        batch = resp.json()
        if not batch:
            break

        all_trades.extend(batch)
        retries = 0

        last_ts_ms = int(pd.Timestamp(batch[-1]["timestamp"]).timestamp() * 1000)

        if len(batch) < MAX_LIMIT or last_ts_ms >= till_ts_ms:
            break

        current_from = last_ts_ms + 1
        time.sleep(REQUEST_SLEEP)

    return all_trades


def trades_to_table(trades, coin):
    """
    Converts raw trade dicts into a de-duplicated PyArrow Table matching
    the fixed SCHEMA, ready to be written as a Parquet file.
    """
    df = pd.DataFrame(trades)
    df["price"] = df["price"].astype(float)
    df["qty"] = df["qty"].astype(float)
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df["coin"] = coin.lower()
    df = df[["id", "price", "qty", "side", "timestamp", "coin"]]
    df = df.drop_duplicates(subset="id").sort_values("timestamp").reset_index(drop=True)
    return pa.Table.from_pandas(df, schema=SCHEMA, preserve_index=False)


def write_day_file(table, coin, day):
    """
    Writes one day's worth of trades to its own Parquet file. Never
    overwrites an existing day file unless explicitly told to (used for
    resume: a day is only ever downloaded once).
    """
    coin_dir = os.path.join(OUTPUT_DIR, coin.lower())
    os.makedirs(coin_dir, exist_ok=True)
    filepath = os.path.join(coin_dir, f"{coin.lower()}_{day.date()}.parquet")
    pq.write_table(table, filepath, compression="snappy")
    return filepath


def day_file_exists(coin, day):
    coin_dir = os.path.join(OUTPUT_DIR, coin.lower())
    filepath = os.path.join(coin_dir, f"{coin.lower()}_{day.date()}.parquet")
    return os.path.exists(filepath)


def get_existing_days(coin):
    """Returns a sorted list of dates already downloaded for this coin."""
    coin_dir = os.path.join(OUTPUT_DIR, coin.lower())
    if not os.path.isdir(coin_dir):
        return []
    days = []
    for f in os.listdir(coin_dir):
        if f.endswith(".parquet"):
            date_str = f.replace(f"{coin.lower()}_", "").replace(".parquet", "")
            try:
                days.append(datetime.strptime(date_str, "%Y-%m-%d").date())
            except ValueError:
                continue
    return sorted(days)


def daterange(start, end):
    current = start
    while current < end:
        yield current
        current += timedelta(days=1)

# ----------------------------------------------------------------------------
# Per-coin download for one day
# ----------------------------------------------------------------------------

def download_one_day(coin, symbol, day, session):
    """
    Downloads and writes exactly one day of trades for one coin, unless
    that day's file already exists (resume support).
    """
    if day_file_exists(coin, day):
        log.info(f"{coin}: {day.date()} already downloaded, skipping")
        return 0

    from_ms = int(day.timestamp() * 1000)
    till_ms = int((day + timedelta(days=1)).timestamp() * 1000)

    trades = fetch_trades_for_window(symbol, from_ms, till_ms, session)

    if not trades:
        log.info(f"{coin}: no trades for {day.date()}")
        return 0

    table = trades_to_table(trades, coin)
    filepath = write_day_file(table, coin, day)
    log.info(f"{coin}: saved {len(trades)} trades for {day.date()} -> {filepath}")
    time.sleep(REQUEST_SLEEP)
    return len(trades)

# ----------------------------------------------------------------------------
# Pipeline modes
# ----------------------------------------------------------------------------

def run_backfill(start_date=START_DATE, end_date=None):
    """
    Historical backfill: downloads every missing day, per coin, from
    start_date up to (but not including) end_date. Safe to re-run — already
    downloaded days are skipped automatically.
    """
    end_date = end_date or datetime.now(timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0)
    session = requests.Session()

    for coin in COINS:
        symbol = resolve_symbol(coin, session)
        if symbol is None:
            continue

        log.info(f"=== Backfilling {symbol} from {start_date.date()} to {end_date.date()} ===")
        total = 0
        for day in daterange(start_date, end_date):
            total += download_one_day(coin, symbol, day, session)
        log.info(f"=== {coin}: backfill complete, {total} new trades ===")


def run_daily_update():
    """
    Incremental update: for each coin, finds the most recent day already
    downloaded and fetches every day from there through yesterday (today is
    skipped since it is still in progress and would produce a partial file).
    """
    session = requests.Session()
    yesterday = (datetime.now(timezone.utc) - timedelta(days=1)).replace(
        hour=0, minute=0, second=0, microsecond=0
    )

    for coin in COINS:
        symbol = resolve_symbol(coin, session)
        if symbol is None:
            continue

        existing_days = get_existing_days(coin)
        if existing_days:
            resume_from = datetime.combine(
                existing_days[-1] + timedelta(days=1), datetime.min.time()
            ).replace(tzinfo=timezone.utc)
        else:
            resume_from = START_DATE

        if resume_from > yesterday:
            log.info(f"{coin}: already up to date")
            continue

        log.info(f"=== Updating {symbol} from {resume_from.date()} through {yesterday.date()} ===")
        total = 0
        for day in daterange(resume_from, yesterday + timedelta(days=1)):
            total += download_one_day(coin, symbol, day, session)
        log.info(f"=== {coin}: update complete, {total} new trades ===")

# ----------------------------------------------------------------------------
# Entry point
# ----------------------------------------------------------------------------

if __name__ == "__main__":
    mode = sys.argv[1] if len(sys.argv) > 1 else "backfill"

    if mode == "backfill":
        run_backfill()
    elif mode == "update":
        run_daily_update()
    else:
        print("Usage: python hitbtc_pipeline.py [backfill|update]")
        sys.exit(1)